In [0]:
!pip install shap

In [0]:
import pprint
import yaml
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from modelling.modelling import (
    build_random_forest,
    compute_shap_values,
    compute_shap_interactions,
    get_shap_contributions,
    create_feature_summary,
)
from modelling.modelling_utils import calculate_vif, run_ols_for_coefficients

# Load configs
with open('../configs/models.yaml', 'r') as f:
    model_config = yaml.safe_load(f)

with open('../configs/data_paths.yaml', 'r') as f:
    paths_config = yaml.safe_load(f)

pd.set_option('display.max_columns', None)
print("Imports and configs loaded successfully")

In [0]:
# Loading user level dataframe
notebook_path = os.getcwd()
repo_root = os.path.abspath(os.path.join(notebook_path, ".."))
misc_dir = os.path.join(repo_root, "misc")

user_df_input_path = os.path.join(misc_dir,
                           os.path.basename(paths_config['output_files']['user_info_df_post_eda']))

test_results_df_input_path = os.path.join(misc_dir,
                           os.path.basename(paths_config['output_files']['test_results_df']))

user_info_df = pd.read_parquet(user_df_input_path)
test_results_df = pd.read_parquet(test_results_df_input_path)

user_info_df = user_info_df[~user_info_df['wonky_study_count'].isna()]
print(f"Data loaded: {user_info_df.shape}")

In [0]:
import mlflow
mlflow.autolog(disable=True)

In [0]:
break

#### Stage 1 - VIF & Correlation checks

Although RF is fairly robust against multicolinearity it is still a good idea to check corrrelations for model trade offs.

Some models may have slightly lower explanatory power but better interprepattion.

In [0]:
sig_vif_detailed = calculate_vif_by_feature_set(user_info_df, sig_feature_set_df)

sig_vif_detailed.display()

In [0]:
# feature list
feature_list = sig_feature_set_df["feature"].tolist()

In [0]:
# manual list of vars to exclude (review VIF table and pick highlight correlated vars with groups.)
vars_to_rem = ['ditr_os_android', 'exposure_band_control', 'is_weekend', 'is_evening', 'is_afternoon']
neutral_vars_to_rem ['taskTitle_Awesome advertising task!', 'exposure_band_exposed', 'is_friday']

# removal
updated_feature_list = [i for i in feature_list if i not in vars_to_rem]
updated_feature_list = [i for i in feature_list if i not in vars_to_rem]

# adding quality and risk values
updated_feature_list = updated_feature_list + ['quality', 'risk']

In [0]:
results = run_random_forest_pipeline(
    user_info_df,
    feature_cols=updated_feature_list,
    do_gridsearch=True,
    gridsearch_param_grid=model_config['random_forest']['gridsearch']['param_grid']
)

In [0]:
results['comparison']

#### SHAP ANALAYSIS

SHAP (SHapley Additive exPlanations) provides:
- Direction of feature effects (not just magnitude)
- Per-observation explanations
- Better feature importance than MDI (Mean Decrease in Impurity)

In [0]:
results = run_shap_analysis(
    results=results,
    df=user_info_df,
    sample_size=1000,
)

In [0]:
# Beeswarm plot - shows direction and distribution
plot_shap_summary(results, max_display=20)

In [0]:
# Bar plot - simpler view of importance
plot_shap_bar(results, max_display=20)

In [0]:
# Dependence plot for top feature
top_feature = results['comparison'].iloc[0]['feature']
plot_shap_dependence(results, feature=top_feature)

#### Feature Interactions

In [0]:
# Analyze feature interactions
interaction_df = analyze_feature_interactions(
    results=results,
    df=user_info_df,
    top_n=10,
    sample_size=500,
)

In [0]:
interaction_df

In [0]:
strong_interactions = list(zip(
    interaction_df[interaction_df['interpretation'] == "Strong"]['feature_1'].tolist(), 
    interaction_df[interaction_df['interpretation'] == "Strong"]['feature_2'].tolist()
))

# Create interaction features
for feat1, feat2 in strong_interactions:
    interaction_name = f"{feat1}_X_{feat2}"
    
    user_info_df[interaction_name] = user_info_df[feat1] * user_info_df[feat2]
    
print(f"Created {len(strong_interactions)} interaction features")

interaction_cols = [col for col in user_info_df.columns if '_X_' in col]
pprint.pprint(f"New columns: {interaction_cols}")

#### Results Summary & Export

In [0]:
# Print model summary
print_model_summary(results)

In [0]:
# Create presentation-ready report
report_df = create_report(results)
display(report_df.head(100))

In [0]:
# Generate executive summary
summary = generate_executive_summary(results, outcome_description="wonkiness")

In [0]:
# Visualize top features with RF importance and SHAP direction
top_20 = results['comparison'].head(20).copy()

fig = go.Figure()

# RF importance bars
colors = ['#2ecc71' if d == '↓ Decreases' else '#e74c3c' if d == '↑ Increases' else '#95a5a6' 
          for d in top_20['shap_direction']]

fig.add_trace(go.Bar(
    x=top_20['rf_importance_pct'],
    y=top_20['feature'],
    orientation='h',
    name='RF Importance (%)',
    marker=dict(color=colors),
    text=top_20['shap_direction'],
    textposition='outside',
))

fig.update_layout(
    title="Top 20 Features: RF Importance with SHAP Direction",
    xaxis_title='RF Importance (%)',
    yaxis_title='Feature',
    yaxis=dict(autorange='reversed'),
    height=600,
    margin=dict(l=200, r=100),
    showlegend=False,
)

fig.show()

In [0]:
# Export results to CSV
export_df = export_results(results, output_path='feature_importance_results.csv')